# Rossmann Sales EDA and Feature Engineering

This notebook analyzes sales trends, seasonality, promotions, holidays, and store segments. It then builds the model-ready feature dataset and saves it to `data/processed/features.parquet`.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from src.features import build_features

FIGURE_DIR = PROJECT_ROOT / 'reports' / 'figures'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
def load_analysis_data():
    """Load a prepared local file first, otherwise query PostgreSQL."""
    local_candidates = [
        PROCESSED_DIR / 'daily_sales_with_store.csv',
        PROCESSED_DIR / 'daily_sales_with_store.parquet',
    ]
    for path in local_candidates:
        if path.exists():
            return pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_csv(path)

    load_dotenv(PROJECT_ROOT / '.env')
    required = ['POSTGRES_HOST', 'POSTGRES_PORT', 'POSTGRES_DB', 'POSTGRES_USER', 'POSTGRES_PASSWORD']
    missing = [name for name in required if not os.getenv(name)]
    if missing:
        raise RuntimeError(
            'No local prepared dataset or PostgreSQL configuration found. '
            f"Missing environment variables: {', '.join(missing)}"
        )

    url = URL.create(
        'postgresql+psycopg2',
        username=os.environ['POSTGRES_USER'],
        password=os.environ['POSTGRES_PASSWORD'],
        host=os.environ['POSTGRES_HOST'],
        port=int(os.environ['POSTGRES_PORT']),
        database=os.environ['POSTGRES_DB'],
    )
    query = '''
        SELECT
            d.store_id, d.sales_date, d.sales, d.customers, d.open, d.promo,
            d.state_holiday, d.school_holiday,
            m.store_type, m.assortment, m.competition_distance,
            m.competition_open_since_month, m.competition_open_since_year,
            m.promo2, m.promo2_since_week, m.promo2_since_year, m.promo_interval
        FROM daily_sales AS d
        JOIN store_metadata AS m USING (store_id)
        ORDER BY d.store_id, d.sales_date
    '''
    return pd.read_sql(query, create_engine(url, pool_pre_ping=True))

df = load_analysis_data()
df['sales_date'] = pd.to_datetime(df['sales_date'])
df['state_holiday'] = df['state_holiday'].fillna('0').astype(str).str.lower()
df.info()
df.head()

In [ ]:
# Data quality checks
assert not df.duplicated(['store_id', 'sales_date']).any()
assert df['sales'].ge(0).all()
summary = pd.Series({
    'rows': len(df),
    'stores': df['store_id'].nunique(),
    'start_date': df['sales_date'].min(),
    'end_date': df['sales_date'].max(),
    'total_sales': df['sales'].sum(),
    'average_daily_store_sales': df['sales'].mean(),
})
summary

In [ ]:
# 1. Overall daily sales trend with a 28-day moving average
daily = df.groupby('sales_date', as_index=False)['sales'].sum().sort_values('sales_date')
daily['sales_ma_28'] = daily['sales'].rolling(28, min_periods=1).mean()
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily['sales_date'], daily['sales'], color='#9ecae1', alpha=0.45, linewidth=0.8, label='Daily sales')
ax.plot(daily['sales_date'], daily['sales_ma_28'], color='#08519c', linewidth=2, label='28-day average')
ax.set(title='Rossmann Sales Trend', xlabel='', ylabel='Total sales')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / '01_overall_sales_trend.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# 2. Weekly and monthly seasonality
seasonal = df.assign(
    day_name=df['sales_date'].dt.day_name(),
    month=df['sales_date'].dt.month,
)
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly = seasonal.groupby('day_name')['sales'].mean().reindex(day_order)
monthly = seasonal.groupby('month')['sales'].mean()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=weekly.index, y=weekly.values, ax=axes[0], color='#3182bd')
axes[0].tick_params(axis='x', rotation=35)
axes[0].set(title='Average Sales by Day of Week', xlabel='', ylabel='Average store sales')
sns.lineplot(x=monthly.index, y=monthly.values, marker='o', ax=axes[1], color='#e6550d')
axes[1].set(title='Average Sales by Month', xlabel='Month', ylabel='Average store sales', xticks=range(1, 13))
fig.tight_layout()
fig.savefig(FIGURE_DIR / '02_weekly_monthly_seasonality.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# 3. Promotion sales lift
promo_summary = df.groupby('promo')['sales'].agg(['mean', 'median', 'count'])
promo_lift_pct = 100 * (promo_summary.loc[1, 'mean'] / promo_summary.loc[0, 'mean'] - 1)
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=df, x='promo', y='sales', estimator=np.mean, errorbar=('ci', 95), ax=ax, palette=['#9ecae1', '#08519c'])
ax.set(title=f'Promotion-Day Sales Lift: {promo_lift_pct:.1f}%', xlabel='Promotion active', ylabel='Average store sales')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '03_promotion_sales_lift.png', dpi=160, bbox_inches='tight')
plt.show()
promo_summary

In [ ]:
# 4. State and school holiday effects
holiday_plot = df.assign(
    state_holiday_label=df['state_holiday'].map({'0': 'None', 'a': 'Public', 'b': 'Easter', 'c': 'Christmas'}).fillna(df['state_holiday'])
)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=holiday_plot, x='state_holiday_label', y='sales', estimator=np.mean, errorbar=None, ax=axes[0], color='#756bb1')
axes[0].set(title='Average Sales by State-Holiday Type', xlabel='', ylabel='Average store sales')
sns.barplot(data=holiday_plot, x='school_holiday', y='sales', estimator=np.mean, errorbar=None, ax=axes[1], color='#31a354')
axes[1].set(title='Average Sales: School Holiday vs Regular Day', xlabel='School holiday', ylabel='Average store sales')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '04_holiday_effects.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# 5. Store type and assortment differences
segment = df.groupby(['store_type', 'assortment'], as_index=False).agg(
    average_sales=('sales', 'mean'),
    average_customers=('customers', 'mean'),
    observations=('sales', 'size'),
)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=segment, x='store_type', y='average_sales', hue='assortment', ax=ax)
ax.set(title='Average Sales by Store Type and Assortment', xlabel='Store type', ylabel='Average store sales')
fig.tight_layout()
fig.savefig(FIGURE_DIR / '05_store_type_assortment.png', dpi=160, bbox_inches='tight')
plt.show()
segment.sort_values('average_sales', ascending=False)

In [ ]:
# Build and persist the model-ready feature table.
features = build_features(df)
feature_path = PROCESSED_DIR / 'features.parquet'
features.to_parquet(feature_path, index=False)
print(f'Saved {len(features):,} rows and {features.shape[1]} columns to {feature_path}')
features.head()

In [ ]:
# Data-backed findings for the final report.
best_day = weekly.idxmax()
best_month = int(monthly.idxmax())
best_segment = segment.loc[segment['average_sales'].idxmax()]
print(f'1. Promo days average {promo_lift_pct:.1f}% higher sales than non-promo days.')
print(f'2. {best_day} has the highest average weekly sales pattern; month {best_month} has the highest monthly average.')
print(f"3. Store type {best_segment['store_type']} with assortment {best_segment['assortment']} has the highest average sales among the observed segments.")